In [0]:
%sql

select * from databrickslearning.silver.fact_visit

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df =spark.read.table("databrickslearning.silver.fact_visit")

In [0]:
w = (
    Window
      .partitionBy("patient_id")
      .orderBy("admission_date")
)

df_with_prev =(
    df.withColumn("prev_discharge_date", lag("discharge_date").over(w))
    .withColumn("days_since_last_visit", datediff("admission_date", "prev_discharge_date"))
    .withColumn("is_readmission_30d",when(col("days_since_last_visit")<=30,1).otherwise(0))
    
)

display(df_with_prev)


In [0]:
gold_df =(
    df_with_prev.groupBy("hospital_id","diagnosis_code","hospital_name")
    .agg(count("*").alias("total_visits"),
         sum("is_readmission_30d").alias("total_readmissions"),
         round(sum("is_readmission_30d")/count("*"),3).alias("readmission_rate"),
         sum("cost").alias("total_cost"),
        avg("cost").alias("avg_cost"))
    .withColumn("gold_load_time",current_timestamp())
    
)

         
    

In [0]:
display(gold_df)

In [0]:
gold_df.write.mode("overwrite").saveAsTable("databrickslearning.gold.hospital_disease_kpi")

Which hospital has the highest readmission rate?

In [0]:

%sql

SELECT hospital_name, readmission_rate
FROM databrickslearning.gold.hospital_disease_kpi
ORDER BY readmission_rate DESC;

Which disease category causes maximum readmissions?

In [0]:
%sql
SELECT diagnosis_code, SUM(total_readmissions) AS readm
FROM databrickslearning.gold.hospital_disease_kpi
GROUP BY diagnosis_code
ORDER BY readm DESC;